# 001 · Importing Modules

> **Syntax › Modules & Packages** · Python 3.14

In Python every `.py` file is a **module**, and `import` is how you use code from another one. It looks like C#'s `using`, but it does much more: an `import` statement actually **finds, loads and runs** code at runtime.

**In this notebook you will learn:**

- what a module is, compared with C# namespaces and assemblies
- `import x`, `from x import y`, and `as` aliases vs `using` / `using static` / `using Alias =`
- how modules are cached in `sys.modules` and found via `sys.path`
- why `from x import *` is discouraged, and how to explore a module with `dir()`
- conditional and deferred (function-level) imports, and the circular-import trap

## C# mental model

| C# | Python |
|---|---|
| namespace | module (one `.py` file) or package (a folder of modules) |
| assembly (`.dll`) + `<PackageReference>` | installed **distribution** (from PyPI) containing packages |
| `using System.Text.Json;` | `import json` |
| `using static System.Math;` | `from math import sqrt, pi` |
| `using Json = System.Text.Json.JsonSerializer;` | `import json as js` / `from json import dumps as to_json` |
| compile-time only — no code runs | **runtime** statement — the module's top-level code executes |
| `Assembly.Load` / reflection | `importlib.import_module("name")` |

The key mental shift: `using` just shortens names for the compiler. Python's `import` is an executable statement that loads the module (once), runs its top-level code, and **binds a name** in your current namespace.

## `import module`

The simplest form binds the module object to a name. You then access its contents with a dot — like a fully-qualified name in C#.

```csharp
// C#
using System;
Console.WriteLine(Math.Sqrt(16));
```

In [ ]:
import math

print(math.sqrt(16))
print(math.pi)
print(type(math))       # modules are objects too
print(math.__name__)

Submodules of a package are imported with dotted names:

In [ ]:
import os.path
import urllib.parse

print(os.path.join("data", "orders.csv"))
print(urllib.parse.urlparse("https://example.com/products?id=7").query)

## `from module import name`

Pulls specific names into your namespace, so you can use them unqualified — similar to `using static`.

```csharp
// C#
using static System.Math;
Console.WriteLine(Sqrt(16) + PI);
```

In [ ]:
from math import pi, sqrt
from collections import Counter, defaultdict

print(sqrt(16) + pi)
print(Counter("banana").most_common(1))

## Aliases with `as`

```csharp
// C#
using Json = System.Text.Json.JsonSerializer;
var text = Json.Serialize(new { id = 1 });
```

In [ ]:
import datetime as dt                   # module alias
from json import dumps as to_json       # name alias

print(to_json({"id": 1, "tags": ["new"]}))
print(dt.date(2024, 1, 15).isoformat())

💡 Some aliases are community conventions you'll see everywhere (e.g. `import numpy as np`, `import pandas as pd` for third-party libraries). Otherwise, only alias when it genuinely improves readability or avoids a name clash.

Imports should go at the top of the file, grouped as **standard library**, then **third-party**, then **your own modules** — tools like `ruff` or `isort` sort them automatically (≈ `dotnet format` sorting usings).

```python
import json
import sys
from pathlib import Path

import requests            # third-party

from shop.models import Product   # your own package
```

## Module caching: `sys.modules`

A module is loaded **once per process**. The first `import` runs it and stores it in the `sys.modules` dictionary; every later `import` of the same name just returns the cached object. (C# assemblies are similarly loaded once per AppDomain/AssemblyLoadContext.)

In [ ]:
import sys
import json

print("json" in sys.modules)
print(sys.modules["json"] is json)   # same object

import json as again                 # no reload — just another name for the cached module
print(again is json)

To see that top-level code really runs only once, let's create a tiny module in a temporary directory (we'll remove it at the end of the notebook):

In [ ]:
import sys
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
tmp_path = Path(tmp.name)
(tmp_path / "greeter.py").write_text(
    'print("  greeter: top-level code running!")\n'
    'def hello(name):\n'
    '    return f"hello, {name}"\n'
)
sys.path.insert(0, str(tmp_path))   # make the folder importable

import greeter          # runs the module
import greeter          # cached: no output this time
print(greeter.hello("alice"))

`importlib.reload()` forces a re-run — handy in notebooks after you edit a module file, but rarely used in application code.

In [ ]:
import importlib

greeter = importlib.reload(greeter)   # runs the top-level code again

## Where Python looks: `sys.path`

When you `import x`, Python searches the directories in `sys.path` in order — roughly: the script's folder (or the current directory), the standard library, then the environment's `site-packages` (where installed packages live). Jupyter orders the entries slightly differently, and our temporary folder is first because we inserted it at position 0. This is the runtime counterpart of MSBuild resolving assembly references.

In [ ]:
from pathlib import Path

home = str(Path.home())
for entry in sys.path[:6]:
    label = entry.replace(str(tmp_path), "<our temp folder>").replace(home, "~")
    print(" ", label or "<current directory>")
print(f"  ... {len(sys.path)} entries in total")

You can find out which file a module was loaded from with `__file__`:

In [ ]:
print(Path(json.__file__).parent.name, "/", Path(json.__file__).name)   # json is a package
print(Path(greeter.__file__).name)

⚠️ **Gotcha:** because the script's own folder comes first, a file named like a standard module **shadows** it. A file called `json.py`, `random.py` or `email.py` next to your script breaks `import json` etc. Avoid naming your modules after standard-library modules.

## Exploring a module: `dir()`

`dir(module)` lists the names a module defines — a quick substitute for "Go to Definition" or the Object Browser. `help(module)` prints its documentation.

In [ ]:
print([name for name in dir(math) if not name.startswith("_")][:12])
print(dir(greeter))

## ⚠️ Avoid `from module import *`

A star import dumps every public name of the module into your namespace. It hides where names come from and can silently overwrite your own names — or built-ins.

In [ ]:
def floor(x):            # our own helper
    return "my floor"


from math import *       # silently replaces our floor (and adds ~60 other names)

print(floor(2.7))        # 2 — math.floor, not ours!

Write explicit imports instead (`from math import sqrt, pi`) or import the module (`import math`). Star imports are acceptable only in an interactive session. A module can limit what `*` exports with `__all__` *(see Modules & Packages › 002 · Creating Modules and Packages)*.

## Dynamic imports

`importlib.import_module()` imports a module whose name is only known at runtime — useful for plugin systems (≈ `Assembly.Load` + reflection).

In [ ]:
import importlib

for name in ("json", "csv", "not_a_real_module"):
    try:
        mod = importlib.import_module(name)
        print(f"loaded {mod.__name__}")
    except ModuleNotFoundError as e:
        print(f"{type(e).__name__}: {e}")

## Conditional imports

Since `import` is just a statement, it can live inside `try`/`except` or `if` — a common pattern for optional dependencies or version differences. (C# would use `#if` symbols or runtime feature checks.)

In [ ]:
try:
    import tomllib                    # standard library since 3.11
except ModuleNotFoundError:
    tomllib = None                    # optional feature disabled

if sys.version_info >= (3, 12):
    from itertools import batched
else:
    batched = None

print(tomllib is not None, batched is not None)

## Deferred imports inside functions

Python 3.14 has **no special syntax for lazy imports**. If a module is expensive to load and only needed occasionally, the usual technique is to import it **inside the function** that needs it. The first call pays the cost; later calls hit the `sys.modules` cache.

In [ ]:
def export_report(rows: list[dict]) -> str:
    import csv          # loaded only when a report is actually exported
    import io

    buffer = io.StringIO()
    writer = csv.DictWriter(buffer, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
    return buffer.getvalue()


print(export_report([{"product": "widget", "qty": 3}, {"product": "gadget", "qty": 1}]))

💡 Function-level imports also help break circular imports (next section). Keep them the exception — top-level imports make dependencies visible at a glance.

## Circular imports

C# happily lets two classes in the same assembly reference each other. In Python, if `a.py` imports `b.py` and `b.py` imports `a.py` **at the top level**, one of them sees the other only **partially initialised** — names defined later in the file don't exist yet:

In [ ]:
(tmp_path / "orders.py").write_text(
    "from customers import Customer\n"
    "class Order:\n"
    "    pass\n"
)
(tmp_path / "customers.py").write_text(
    "from orders import Order       # orders is only half-loaded at this point!\n"
    "class Customer:\n"
    "    pass\n"
)

try:
    import orders
except ImportError as e:
    print(f"{type(e).__name__}: {e}")

Typical fixes:

- move the shared code into a third module that both import (best — usually a sign of a design issue)
- import the **module** instead of a name (`import orders` then `orders.Order` at call time)
- move the import into the function that needs it
- for imports needed only in type hints, guard them with `if typing.TYPE_CHECKING:` — and since Python 3.14 evaluates annotations lazily *(see Type Hints › 001 · Type Annotations)*, no quotes are needed

Finally, clean up the temporary modules so the rest of the session isn't affected:

In [ ]:
for name in ("greeter", "orders", "customers"):
    sys.modules.pop(name, None)
sys.path.remove(str(tmp_path))
tmp.cleanup()
print("greeter" in sys.modules, tmp_path.exists())

## Summary

| Task | C# | Python |
|---|---|---|
| Use a namespace/module | `using System.Text.Json;` | `import json` |
| Import specific members | `using static System.Math;` | `from math import sqrt` |
| Alias | `using J = System.Text.Json.JsonSerializer;` | `import json as j` / `from x import y as z` |
| Load by name at runtime | `Assembly.Load(...)` | `importlib.import_module("x")` |
| Already-loaded modules | loaded assemblies | `sys.modules` |
| Search locations | assembly probing paths | `sys.path` |
| List members | Object Browser / reflection | `dir(module)` |
| Import everything | — | `from x import *` (avoid) |
| Lazy loading | `Lazy<T>` / on-demand load | import inside a function |

## Exercises

1. Import `statistics` with an alias `stats` and print the median of `[3, 1, 4, 1, 5]`.
2. Using `importlib.import_module`, write a function `try_import(name)` that returns the module or `None` if it doesn't exist. Test it with `"decimal"` and `"no_such_module"`.
3. Print every public name in the `random` module that starts with `"rand"`.
4. Explain what goes wrong if you save your own script as `random.py` and it contains `import random`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import importlib
import random

# 1
import statistics as stats
print(stats.median([3, 1, 4, 1, 5]))   # 3

# 2
def try_import(name):
    try:
        return importlib.import_module(name)
    except ModuleNotFoundError:
        return None

print(try_import("decimal"), try_import("no_such_module"))

# 3
print([n for n in dir(random) if n.startswith("rand")])

# 4 — the script's folder is first on sys.path, so `import random` finds
#     YOUR random.py (itself) instead of the standard library module.
#     Calls like random.randint then fail with AttributeError.
```

</details>